In [1]:
from datetime import datetime

# Obtém a data e horário atual
agora = datetime.now()

# Exibe formatado
print("Data e horário atual:", agora.strftime("%d/%m/%Y %H:%M:%S"))

Data e horário atual: 23/10/2026 16:08:33


In [2]:
"""
e2_pot_symbolic_llm_exec_finqa_autogen.py

EXPERIMENTO E2 - PoT SIMBOLICO SEM CALCULADORA da ablacao  (VERSAO 4,
construida a partir do E1 v4: E1_MultiStep_LLM_FLOAT_Amostra_v4)

Pergunta cientifica: forcar uma representacao formal (programa simbolico)
ja melhora o planejamento em relacao ao multi-step livre (E1)?

VERSAO 4 -- alinhada com E0 v4 / E1 v4:
    - TRES criterios de acerto, com os mesmos nomes de coluna do E0/E1 v4:
        correct_exato      -> float(extraido) == float(gabarito)  (EM_exato)
        correct            -> _answers_match, tol = 1e-2          (EM)
        correct_escala_pct -> aceita gold x100 ou /100 em perguntas de
                              percentual                (EM_escala_percentual)
    - Ponto flutuante: o LLM Executor e instruido a devolver o "answer" em
      ponto flutuante (mesma frase do E0/E1 v4).
    - Conformidade de formato SEM tratamento (como no E1 v4), agora para as
      duas chamadas:
        plan_json_puro           -> a resposta do Planner e SO o JSON
        executor_json_puro       -> a resposta do Executor e SO o JSON
        executor_answer_float    -> "answer" do Executor ja e um float
        formato_conforme         -> executor_json_puro E executor_answer_float
                                    -> conformidade_formato_pct
    - Plan Accuracy objetiva tambem com escala percentual
      (plan_correct_escala_pct), para separar "plano errado" de "plano certo
      com o resultado em fracao".
    - Semantica das operacoes (acrescimo ao prompt do documento): nova regra
      para perguntas de "change / difference / variation", que pedem
      sub(new, old) e NAO percentage_change. Na execucao do E2 v2, 16 dos
      55 planos errados usaram percentage_change em perguntas de variacao
      absoluta (ex.: "what was the change in r&d expenses from 2016 to
      2017?"). Esta regra vale para E2..E7 (mesmo Planner).

VERSAO 2 (mantida):
    - HISTORY no bloco HISTORY; leitura do ULTIMO JSON completo; metrica
      EM_escala_percentual.
    - Programa simbolico (E2..E7): operacao "value" para consultas diretas,
      add/mul com 2 ou mais argumentos, percentuais como fracao, nomes de
      passos diferentes dos de variaveis e 4 exemplos curtos (consulta, soma,
      variacao percentual, proporcao). A calculadora de referencia segue a
      mesma semantica e da prioridade as variaveis quando um passo repete o
      nome de uma variavel.

Versao LOCAL (via Ollama). O LLM NAO produz diretamente a resposta: ele
gera um programa simbolico (PoT) e, numa segunda chamada, o proprio LLM
interpreta/executa esse programa, como sugerido no documento de ablacao.
Nenhum critico, nenhuma calculadora.

Fluxo (por pergunta):

    Contexto (Question + Text + Table + History)   <- MESMA entrada do E0/E1
        |
        v
    LLM Planner   (system: mesma persona do E0/E1; user: prompt E2)
        |
        v
    PoT JSON  {variables, steps[{name, op, args}], unit}
        |
        v
    LLM Executor  (recebe SO o programa, nao o contexto)
        |
        v
    Resposta

Por que o executor recebe so o programa: no E3 a execucao passa para uma
calculadora Python, que tambem recebe so o programa. Assim E2 e E3 diferem
APENAS em quem executa (LLM vs. Python), que e exatamente a pergunta do E3.

Mantido IDENTICO ao E0 v4 / E1 v4:
    - AutoGen (pyautogen==0.2.35), modelo (llama3:8b), temperatura (0.2),
      CSV/amostra, MAX_ITEMS, timeout
    - system prompt (frase de persona do script de referencia)
    - montagem da entrada (build_fields: QUESTION / TEXT / TABLE / HISTORY)
    - extracao do numero (extract_direct_answer, mesma funcao do E0/E1)
    - os tres criterios de acerto e todas as metricas de resposta (MAE,
      MSE/RMSE, IC de Wilson, unidade, custo), com as mesmas formulas e nomes

O que muda (o componente testado):
    - em vez de responder (E0) ou decompor e responder (E1), o LLM gera um
      PROGRAMA simbolico, e outra chamada ao LLM o executa (2 chamadas por
      pergunta).

Prompt do Planner:
    - prompt E2 do documento de ablacao mantido INTEGRALMENTE;
    - ACRESCIMO necessario: a semantica das operacoes. O documento lista
      "percentage" e "percentage_change" mas nao define a ordem dos
      argumentos. Sem isso o plano e ambiguo e, no E3, a calculadora
      teria de adivinhar uma convencao que o Planner nao conhece;
    - [flag INCLUDE_PLANNER_EXAMPLE] exemplos curtos. O E1 nao tinha
      exemplo; para uma comparacao E1 x E2 estrita, desligue a flag. De E2 a
      E7 o prompt do Planner deve ser o mesmo.

Metricas que so existem a partir do E2 (ha um plano formal e executavel):
    - Conformidade: % de plano em JSON valido, % de plano executavel
      (operacoes permitidas, aridade, referencias resolvidas).
    - Plan Accuracy OBJETIVA: o plano e executado por um executor Python
      de REFERENCIA (so para medicao; o resultado dele NAO e usado como
      resposta). Se o resultado bate com o gabarito, o plano esta certo.
    - Execution Fidelity: entre os planos executaveis, % em que o LLM
      Executor chegou ao mesmo numero que o executor de referencia.
    - Decomposicao dos erros por pergunta (coluna error_category):
      plano_invalido | plano_errado | erro_execucao | correto |
      correto_por_acaso (acertou apesar de plano invalido/errado).
    Essas metricas antecipam a pergunta do E3: quanto do erro vem do plano
    e quanto vem da execucao aritmetica feita pelo LLM.

Comparacoes entre experimentos (Kappa, McNemar) serao feitas depois, num
script separado, entre os CSVs de E0..E7 pareados por question_id, usando
as colunas correct_exato / correct / correct_escala_pct de cada um.

Saidas:
    - OUT_PATH                        -> resultados por pergunta (CSV)
    - *_metricas.csv                  -> metricas agregadas (1 linha)
    - *_per_example.jsonl             -> {"id","gold","prediction","correct",
                                          "latency","llm_calls"} por exemplo

Como rodar localmente:
    1. Instale e rode o Ollama: https://ollama.com
    2. Baixe o modelo:            ollama pull llama3:8b
    3. Confirme que o servidor esta ativo em http://localhost:11434
    4. Instale as dependencias:   pip install "pyautogen==0.2.35" pandas
       (IMPORTANTE: a partir da versao 0.4/pyautogen>=0.9 o pacote virou um
       proxy para a nova arquitetura assincrona "autogen-agentchat". Este
       script usa a API classica sincrona, por isso a versao fixada 0.2.35.)
    5. Ajuste as variaveis de configuracao abaixo se necessario
    6. Rode:                      python e2_pot_symbolic_llm_exec_finqa_autogen.py

Requisitos:
    pip install "pyautogen==0.2.35" pandas
    (opcional) pip install scipy   -> z exato para o Intervalo de Confianca
"""

import json
import math
import re
import sys
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import pandas as pd

try:
    import autogen
    from autogen import AssistantAgent
except ImportError:
    print("Instale o SDK primeiro: pip install \"pyautogen==0.2.35\"", file=sys.stderr)
    raise

# scipy e usado APENAS para o valor "z" exato do Intervalo de Confianca.
# E opcional: se nao estiver instalado, o script usa valores de z
# tabelados (90/95/99%).
try:
    from scipy import stats as scipy_stats
    SCIPY_AVAILABLE = True
except ImportError:
    scipy_stats = None
    SCIPY_AVAILABLE = False


# ---------------------------------------------------------------------------
# CONFIGURACAO - ajuste aqui
# ---------------------------------------------------------------------------

EXPERIMENT_ID = "E2_pot_symbolic_v4"
CSV_PATH = "convfinqa_amostra_100.csv"
OLLAMA_MODEL = "llama3:8b"
OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_API_KEY = "ollama"          # Ollama ignora o valor, mas o SDK exige algum
MAX_ITEMS = 100                    # quantas linhas do CSV serao lidas/processadas
TIMEOUT_SECONDS = 600              # timeout por chamada ao modelo
OUT_PATH = "results_E2_pot_symbolic_autogen_v4.csv"
PER_EXAMPLE_JSONL_PATH = OUT_PATH.replace(".csv", "_per_example.jsonl")
VERBOSE = True

# Mesma temperatura do E0/E1 para o Planner. O Executor usa a mesma
# temperatura, para que a unica diferenca no E3 seja trocar o executor.
LLM_TEMPERATURE = 0.2

# Inclui no prompt do Planner o exemplo do documento (receita 120 -> 150).
# O E1 nao tinha exemplo: para comparar E1 x E2 isolando so o PoT, use False.
# Qualquer que seja a escolha, mantenha-a igual de E2 a E7.
INCLUDE_PLANNER_EXAMPLE = True

# Colunas do CSV. O script de referencia usa apenas "text" (contexto +
# pergunta juntos) e "output" (gabarito). Se o seu CSV tiver colunas
# separadas para pergunta / tabela / historico (ex.: ConvFinQA), informe os
# nomes aqui para preencher cada bloco do prompt. Colunas inexistentes
# sao ignoradas automaticamente (fallback descrito em build_fields).
TEXT_COL = "text"
GOLD_COL = "output"
QUESTION_COL = "question"          # opcional
TABLE_COL = "table"                # opcional
HISTORY_COL = "history"            # opcional (ConvFinQA)

# Preco de referencia (USD por 1 milhao de tokens) usado SOMENTE para simular
# quanto custaria caso essa mesma carga de tokens fosse rodada em uma API
# paga. A execucao real deste script e local/gratuita via Ollama.
REFERENCE_API_PRICING = {
    "Groq - Llama 3.1 8B Instant": {"input_per_1m": 0.05, "output_per_1m": 0.08},
    "OpenAI - GPT-4o mini":        {"input_per_1m": 0.15, "output_per_1m": 0.60},
}


# ---------------------------------------------------------------------------
# Prompts
# ---------------------------------------------------------------------------

# (1) System prompt: a MESMA frase de persona usada no E0/E1. Manter igual
#     em E0..E7 para que ela nao vire uma variavel da ablacao.
SYSTEM_PERSONA = """You are a financial analysis agent specializing in interpreting \
earnings reports and financial statements. Your task is to answer specific \
financial questions based on the given context from financial reports."""

# (2) Prompt E2 do documento de ablacao -- mantido INTEGRALMENTE.
#     As chaves do JSON estao duplicadas ({{ }}) apenas por causa do
#     str.format().
E2_PLANNER_PROMPT = """You are a financial reasoning planner.

Do not calculate the final answer.

Generate a symbolic program capable of answering the question.

Allowed operations:

add
sub
mul
div
percentage
percentage_change

Return only JSON:

{{
  "variables": {{
  }},
  "steps": [
    {{
      "name": "",
      "op": "",
      "args": []
    }}
  ],
  "unit": ""
}}
"""

# (3) ACRESCIMO: semantica das operacoes. O documento lista as operacoes mas
#     nao define argumentos nem convencoes. VERSAO 2 (apos a analise do E2):
#     - "value" (1 argumento): perguntas de consulta direta ("qual era o
#       valor de X em 2013?") nao tinham como ser expressas -- o modelo
#       inventava operacoes ("answer", "same") ou fabricava contas;
#     - add/mul aceitam 2 ou mais argumentos (somas de varios itens);
#     - percentuais como FRACAO decimal, a convencao do ConvFinQA (o
#       gabarito de "36.93%" e 0.3693); antes multiplicava por 100;
#     - nomes de passos nao podem repetir nomes de variaveis.
#     VERSAO 4: regra para "change / difference / variation" -> sub(new, old)
#     com sinal; percentage_change so quando a pergunta pede percentual.
#     E a MESMA semantica que a calculadora (e o LLM Executor do E2) usa.
OPERATION_SEMANTICS = """Operation semantics:
- add(a, b, ...) = a + b + ...   (two or more arguments)
- sub(a, b) = a - b
- mul(a, b, ...) = a * b * ...   (two or more arguments)
- div(a, b) = a / b
- percentage(part, whole) = part / whole
- percentage_change(old, new) = (new - old) / old
- value(a) = a   (one argument; use it when the answer is a value read directly from the context)

Rules:
- "variables" holds the numeric values taken from the context (plain numbers, no units).
- Each argument is a variable name, the name of a previous step, or a numeric constant.
- Step names must be different from variable names.
- The last step must be named "answer".
- Express percentages and ratios as decimal fractions (25% -> 0.25). Multiply by 100 only if the question explicitly asks for it.
- A question about the "change", "difference", "variation", "increase" or "decrease" of a value from one period to another asks for sub(new_value, old_value), with its sign. Use percentage_change only when the question asks for a percentage, percent change or growth rate.
"""

# (4) Exemplos curtos cobrindo os tipos de pergunta mais comuns (usados se
#     INCLUDE_PLANNER_EXAMPLE). VERSAO 2: o exemplo unico do documento
#     (variacao percentual) era copiado em perguntas que nao eram de
#     percentual. O exemplo do documento continua aqui (Example 3), agora
#     com percentage_change e o resultado como fracao.
PLANNER_EXAMPLE = """Examples.

Example 1 -- Question: What was the net revenue in 2019?
Relevant values: net revenue 2019 = 845.2
{{"variables": {{"net_revenue_2019": 845.2}},
  "steps": [{{"name": "answer", "op": "value", "args": ["net_revenue_2019"]}}],
  "unit": "million"}}

Example 2 -- Question: What was the combined total of the three segments?
Relevant values: segment A = 12.5, segment B = 30.1, segment C = 7.4
{{"variables": {{"segment_a": 12.5, "segment_b": 30.1, "segment_c": 7.4}},
  "steps": [{{"name": "answer", "op": "add", "args": ["segment_a", "segment_b", "segment_c"]}}],
  "unit": "million"}}

Example 3 -- Question: Revenue increased from 120 to 150. What was the percentage increase?
{{"variables": {{"old_value": 120, "new_value": 150}},
  "steps": [{{"name": "answer", "op": "percentage_change", "args": ["old_value", "new_value"]}}],
  "unit": "%"}}

Example 4 -- Question: What portion of total expenses was R&D?
Relevant values: R&D = 100, total expenses = 400
{{"variables": {{"rd": 100, "total_expenses": 400}},
  "steps": [{{"name": "answer", "op": "percentage", "args": ["rd", "total_expenses"]}}],
  "unit": "%"}}
"""

# (5) Blocos de entrada -- IDENTICOS ao E0/E1.
INPUT_BLOCKS = """QUESTION:
{question}

TEXT:
{text}

TABLE:
{table}

HISTORY:
{history}
"""

# (6) Prompt do LLM Executor (o documento so diz que, no E2, o proprio LLM
#     interpreta/executa o programa). Recebe SO o programa, como a
#     calculadora do E3. VERSAO 4: pede o "answer" em ponto flutuante, com
#     a mesma frase do E0/E1 v4.
EXECUTOR_PROMPT = """You are a symbolic program executor.

Execute the program below exactly as written, one step at a time, in order.
Do not change the program and do not use any outside information.
Return the final answer in floating point format.

{semantics}
Return only JSON:
{{
  "step_results": {{"<step name>": <number>}},
  "answer": <number>
}}

PROGRAM:
{program}
"""

ALLOWED_OPS = ("add", "sub", "mul", "div", "percentage", "percentage_change", "value")


# ---------------------------------------------------------------------------
# Estruturas de dados
# ---------------------------------------------------------------------------

@dataclass
class PoTResult:
    question_id: object
    ground_truth: str

    # resposta final (do LLM Executor)
    prediction: str = ""
    correct: bool = False               # tol 1e-2 (EM)
    correct_exato: bool = False         # float(extraido) == float(gabarito)
    answer_extraction: str = ""
    parse_error: bool = False       # True se nenhum numero foi obtido

    # Planner
    plan_json_valid: bool = False
    plan_json_puro: bool = False        # resposta do Planner = so o JSON
    plan: str = ""                  # programa normalizado (JSON)
    plan_unit: str = ""
    plan_n_steps: int = 0
    planner_raw: str = ""

    # Executor de REFERENCIA (Python) -- so para medicao
    ref_status: str = ""            # ok | motivo da falha
    ref_answer: Optional[float] = None
    plan_correct: bool = False      # ref_answer bate com o gabarito (tol 1e-2)
    plan_correct_escala_pct: bool = False   # idem, aceitando gold x100 ou /100
    execution_faithful: Optional[bool] = None   # LLM exec == ref (se executavel)
    error_category: str = ""

    # LLM Executor
    executor_called: bool = False
    executor_json_valid: bool = False
    executor_json_puro: bool = False        # resposta do Executor = so o JSON
    executor_answer_float: bool = False     # "answer" do Executor ja e float
    formato_conforme: bool = False          # executor_json_puro E executor_answer_float
    executor_raw: str = ""

    # tokens / custo operacional desta linha
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0
    latency_seconds: float = 0.0
    llm_calls: int = 0


# ---------------------------------------------------------------------------
# Utilitarios de parsing / comparacao de respostas
# ---------------------------------------------------------------------------

def _parse_json_response(text: str) -> dict:
    """Extrai o ULTIMO objeto JSON completo e valido da resposta do modelo.

    Os modelos pequenos costumam escrever texto com varios blocos JSON no
    meio (ex.: resultados parciais e depois o final). A versao anterior
    pegava tudo entre o primeiro "{" e o ultimo "}", o que juntava varios
    blocos num JSON invalido. Agora cada objeto de nivel superior e
    delimitado pelo balanceamento das chaves (ignorando chaves dentro de
    strings) e o ultimo que for valido e devolvido."""
    if text is None:
        return {"raw": "", "parse_error": True}
    candidates = []
    depth, start, in_str, esc = 0, None, False, False
    for i, ch in enumerate(text):
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"' and depth > 0:
            in_str = True
        elif ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}" and depth > 0:
            depth -= 1
            if depth == 0 and start is not None:
                candidates.append(text[start:i + 1])
                start = None
    for cand in reversed(candidates):
        for attempt in (cand, cand.replace("'", '"')):
            try:
                obj = json.loads(attempt)
            except json.JSONDecodeError:
                continue
            if isinstance(obj, dict):
                return obj
    return {"raw": text, "parse_error": True}


def _extract_content(reply) -> str:
    """Extrai com seguranca o texto de uma resposta do AutoGen, que pode vir
    como str, dict ({'content': ...}) ou None (timeout/erro de conexao com
    o Ollama). Retorna sempre uma string (vazia se nao houver conteudo)."""
    if reply is None:
        return ""
    if isinstance(reply, str):
        return reply
    if isinstance(reply, dict):
        return reply.get("content", "") or ""
    return str(reply)


def _to_float(x):
    if x is None:
        return None
    s = str(x).strip().replace(",", "").replace("%", "").replace("$", "")
    try:
        return float(s)
    except ValueError:
        return None


def _answers_match(a, b, tol: float = 1e-2) -> bool:
    fa, fb = _to_float(a), _to_float(b)
    if fa is None or fb is None:
        return str(a).strip() == str(b).strip()
    return abs(fa - fb) <= tol * max(1.0, abs(fb))


def is_pure_float(raw) -> bool:
    """VERSAO 4 -- conformidade de formato, SEM tratamento: a resposta
    inteira (ignorando so espacos nas pontas) e aceita por float()?
    "12.0" -> True; "$12", "12%", "1,200" ou uma frase -> False."""
    try:
        float(str(raw).strip())
        return True
    except (TypeError, ValueError):
        return False


def answers_match_exact(pred, gold) -> bool:
    """VERSAO 4 -- igualdade EXATA, sem tolerancia, entre o numero extraido
    e o gabarito: float(pred) == float(gold)."""
    fp, fg = _to_float(pred), _to_float(gold)
    if fp is None or fg is None:
        return str(pred).strip() == str(gold).strip()
    return fp == fg


def is_pure_json(raw) -> bool:
    """A resposta INTEIRA (ignorando so espacos nas pontas) e um unico
    objeto JSON valido, sem texto antes ou depois?"""
    try:
        return isinstance(json.loads(str(raw).strip()), dict)
    except (TypeError, ValueError):
        return False


def is_float_answer(ans) -> bool:
    """O campo "answer" ja esta em ponto flutuante, SEM tratamento?
    56.0 / 56 / "56.0" -> True;  "56 million", "$56", "12%", None -> False."""
    if ans is None or isinstance(ans, bool):
        return False
    if isinstance(ans, (int, float)):
        return not (isinstance(ans, float) and math.isnan(ans))
    return is_pure_float(ans)


# ---------------------------------------------------------------------------
# METRICAS
# ---------------------------------------------------------------------------
# Operam em cima do DataFrame de resultados (out_df) depois que todas as
# linhas foram processadas. Nao fazem nenhuma chamada ao LLM.
# Colunas usadas: prediction, correct, ground_truth.

def _pair_valid_floats(pred_series: pd.Series, gt_series: pd.Series):
    """Devolve duas listas (pred, gt) so com pares onde AMBOS os valores sao
    numericos validos. Usado para MAE / erro relativo / MSE, que nao fazem
    sentido para respostas nao-numericas ou nao-parseaveis."""
    preds, gts = [], []
    for p, g in zip(pred_series, gt_series):
        fp, fg = _to_float(p), _to_float(g)
        if fp is not None and fg is not None:
            preds.append(fp)
            gts.append(fg)
    return preds, gts


# ---------------------------------------------------------------------------
# EM com tolerancia de escala percentual
# ---------------------------------------------------------------------------
# No ConvFinQA os percentuais estao no gabarito como FRACAO (36.93% = 0.3693),
# mas alguns poucos estao em pontos percentuais. Um modelo que responde
# "36.93" para uma pergunta de percentual acerta o conteudo e erra so a
# convencao. O EM estrito continua sendo o criterio principal; o
# EM_escala_percentual aceita, SOMENTE em perguntas de percentual/razao,
# tambem prediction == gold*100 ou prediction == gold/100. E a metrica
# recomendada para comparar E0..E7, porque nao depende de cada prompt
# conhecer a convencao do dataset.

_PERCENT_Q_RE = re.compile(
    r"(?i)percent|%|\bportion\b|proportion|\bratio\b|represent|return rate|growth rate")


def is_percent_question(question: str) -> bool:
    return bool(_PERCENT_Q_RE.search(str(question or "")))


def answers_match_pct_scale(pred, gold, question: str) -> bool:
    if _answers_match(pred, gold):
        return True
    if not is_percent_question(question):
        return False
    fp, fg = _to_float(pred), _to_float(gold)
    if fp is None or fg is None:
        return False
    return _answers_match(fp, fg * 100) or _answers_match(fp, fg / 100)


def add_question_columns(out_df: pd.DataFrame, subset_df: pd.DataFrame) -> pd.DataFrame:
    """Acrescenta ao CSV a pergunta respondida, se ela e de percentual e o
    acerto com tolerancia de escala percentual."""
    if len(out_df) == 0:
        return out_df
    qs = [build_fields(subset_df.loc[qid])["question"] if qid in subset_df.index else ""
          for qid in out_df["question_id"]]
    out_df.insert(out_df.columns.get_loc("ground_truth"), "question", qs)
    out_df["is_percent_question"] = [is_percent_question(q) for q in qs]
    out_df["correct_escala_pct"] = [answers_match_pct_scale(p, g, q) for p, g, q in
                                    zip(out_df["prediction"], out_df["ground_truth"], qs)]
    return out_df


def compute_answer_level_metrics(out_df: pd.DataFrame) -> Dict[str, float]:
    """Exact Match / Accuracy, MAE e erro relativo medio da resposta direta,
    comparada ao gabarito (ground_truth)."""
    metrics: Dict[str, float] = {}
    n = len(out_df)
    if n == 0:
        return metrics

    metrics["EM"] = 100.0 * out_df["correct"].sum() / n
    if "correct_exato" in out_df.columns:
        metrics["EM_exato"] = 100.0 * out_df["correct_exato"].sum() / n
    if "formato_numero_puro" in out_df.columns:
        metrics["conformidade_formato_pct"] = 100.0 * out_df["formato_numero_puro"].sum() / n
    if "correct_escala_pct" in out_df.columns:
        metrics["EM_escala_percentual"] = 100.0 * out_df["correct_escala_pct"].sum() / n
        metrics["n_perguntas_percentual"] = int(out_df["is_percent_question"].sum())

    preds, gts = _pair_valid_floats(out_df["prediction"], out_df["ground_truth"])
    if preds:
        abs_errors = [abs(p - g) for p, g in zip(preds, gts)]
        rel_errors = [abs(p - g) / max(1.0, abs(g)) for p, g in zip(preds, gts)]
        metrics["MAE"] = sum(abs_errors) / len(abs_errors)
        metrics["ErroRelativoMedio"] = 100.0 * sum(rel_errors) / len(rel_errors)
    else:
        metrics["MAE"] = float("nan")
        metrics["ErroRelativoMedio"] = float("nan")
    metrics["n_pares_numericos"] = len(preds)
    return metrics


def compute_squared_error_metrics(out_df: pd.DataFrame) -> Dict[str, float]:
    """MSE (Erro Quadratico Medio) e RMSE (sua raiz) da resposta direta.

    Complementa o MAE: o MSE eleva os erros ao quadrado antes de tirar a
    media, entao penaliza mais fortemente respostas que erram "feio" (ex.:
    o modelo erra a ordem de grandeza), enquanto o MAE trata todo desvio
    de forma linear."""
    preds, gts = _pair_valid_floats(out_df["prediction"], out_df["ground_truth"])
    if not preds:
        return {"MSE": float("nan"), "RMSE": float("nan"), "n_pares_numericos_mse": 0}
    mse = sum((p - g) ** 2 for p, g in zip(preds, gts)) / len(preds)
    return {"MSE": mse, "RMSE": math.sqrt(mse), "n_pares_numericos_mse": len(preds)}


def _z_for_confidence(confidence: float) -> float:
    """Valor critico z (distribuicao normal padrao, bicaudal) para o nivel
    de confianca pedido. Usa scipy quando disponivel (qualquer confianca);
    caso contrario, cai para uma tabela com os niveis mais comuns."""
    if SCIPY_AVAILABLE:
        return float(scipy_stats.norm.ppf(1 - (1 - confidence) / 2))
    z_table = {0.90: 1.6449, 0.95: 1.9600, 0.99: 2.5758}
    return z_table.get(round(confidence, 2), 1.9600)  # default: 95%


def _wilson_confidence_interval(successes: int, n: int, confidence: float = 0.95) -> Tuple[float, float]:
    """Intervalo de Confianca de Wilson para uma proporcao (EM/Accuracy).

    Preferido ao intervalo normal/Wald classico porque continua confiavel
    com amostras pequenas (ex.: n=100) e quando a proporcao observada esta
    perto de 0% ou 100% -- casos em que o intervalo de Wald pode extrapolar
    para fora de [0%, 100%]."""
    if n == 0:
        return float("nan"), float("nan")
    z = _z_for_confidence(confidence)
    p = successes / n
    denom = 1 + (z ** 2) / n
    centre = p + (z ** 2) / (2 * n)
    margin = z * math.sqrt((p * (1 - p) / n) + (z ** 2) / (4 * n ** 2))
    low = (centre - margin) / denom
    high = (centre + margin) / denom
    return max(0.0, low), min(1.0, high)


def compute_confidence_interval(out_df: pd.DataFrame, confidence: float = 0.95) -> Dict[str, float]:
    """Intervalo de Confianca (Wilson, 95% por padrao) para o EM.

    Com amostras pequenas (ex.: n=100), duas taxas de EM proximas (ex.:
    48% vs 45% entre E0 e E1) podem nao ser estatisticamente diferentes --
    o IC ajuda a visualizar essa incerteza."""
    n = len(out_df)
    if n == 0:
        return {}
    low, high = _wilson_confidence_interval(int(out_df["correct"].sum()), n, confidence)
    metrics = {
        "EM_IC_low": 100.0 * low,
        "EM_IC_high": 100.0 * high,
        "EM_IC_margem": 100.0 * (high - low) / 2,
        "IC_nivel_confianca": confidence,
        "IC_n_amostras": n,
    }
    if "correct_escala_pct" in out_df.columns:
        low, high = _wilson_confidence_interval(int(out_df["correct_escala_pct"].sum()), n, confidence)
        metrics["EM_escala_pct_IC_low"] = 100.0 * low
        metrics["EM_escala_pct_IC_high"] = 100.0 * high
    if "correct_exato" in out_df.columns:
        low, high = _wilson_confidence_interval(int(out_df["correct_exato"].sum()), n, confidence)
        metrics["EM_exato_IC_low"] = 100.0 * low
        metrics["EM_exato_IC_high"] = 100.0 * high
    return metrics


def compute_unit_heuristic_accuracy(subset_df: pd.DataFrame, out_df: pd.DataFrame) -> Dict[str, float]:
    """Aproximacao HEURISTICA de Unit Accuracy (nao usa LLM): para perguntas
    cujo enunciado sugere uma unidade (%, milhao, bilhao), verifica se a
    resposta esta na mesma ordem de grandeza do gabarito (desvio relativo
    < 50%). Um erro de unidade (ex.: 0.25 em vez de 25%) cai fora dessa
    faixa. E um proxy grosseiro.

    As linhas sao pareadas por question_id (indice do CSV), entao linhas
    que falharam durante a execucao nao desalinham a comparacao."""
    def unit_hint(text: str) -> Optional[str]:
        t = str(text).lower()
        if "%" in t or "percent" in t:
            return "percent"
        if "billion" in t:
            return "billion"
        if "million" in t:
            return "million"
        return None

    matches = 0
    total = 0
    for _, row in out_df.iterrows():
        qid = row["question_id"]
        if qid not in subset_df.index:
            continue
        if unit_hint(subset_df.at[qid, TEXT_COL]) is None:
            continue
        total += 1
        fa, fg = _to_float(row["prediction"]), _to_float(row["ground_truth"])
        if fa is None or fg is None:
            continue
        if fg != 0 and abs(fa - fg) / max(1.0, abs(fg)) < 0.5:
            matches += 1
    return {
        "unit_accuracy_heuristica": 100.0 * matches / total if total else float("nan"),
        "n_perguntas_com_unidade_detectada": total,
    }


def compute_plan_metrics(out_df: pd.DataFrame) -> Dict[str, float]:
    """Metricas que so existem a partir do E2 (ha um plano formal):

    - plano_json_valido_pct / plano_executavel_pct: conformidade do Planner.
    - plan_accuracy_objetiva: % de TODAS as perguntas cujo plano, executado
      pelo executor Python de referencia, bate com o gabarito.
    - plan_accuracy_escala_pct: idem, aceitando gold x100 ou /100 em
      perguntas de percentual.
    - conformidade de formato SEM tratamento (como no E1 v4): Planner e
      Executor respondem so com o JSON? "answer" do Executor ja e float?
    - execution_fidelity: entre os planos executaveis com resposta do LLM,
      % em que o LLM Executor chegou ao mesmo numero que a referencia.
    - distribuicao de error_category (em % de todas as perguntas).
    - executor_json_valido_pct: entre as chamadas ao Executor.
    """
    n = len(out_df)
    if n == 0:
        return {}
    executable = out_df["ref_status"] == "ok"
    metrics: Dict[str, float] = {
        "plano_json_valido_pct": 100.0 * out_df["plan_json_valid"].sum() / n,
        "plano_executavel_pct": 100.0 * executable.sum() / n,
        "plan_accuracy_objetiva": 100.0 * out_df["plan_correct"].sum() / n,
        "plan_accuracy_escala_pct": 100.0 * out_df["plan_correct_escala_pct"].sum() / n,
        "plano_json_puro_pct": 100.0 * out_df["plan_json_puro"].sum() / n,
        "executor_json_puro_pct": 100.0 * out_df["executor_json_puro"].sum() / n,
        "executor_answer_float_pct": 100.0 * out_df["executor_answer_float"].sum() / n,
        "conformidade_formato_pct": 100.0 * out_df["formato_conforme"].sum() / n,
    }
    faithful = out_df["execution_faithful"].dropna()
    metrics["execution_fidelity"] = 100.0 * faithful.astype(bool).sum() / len(faithful) if len(faithful) else float("nan")
    metrics["n_execution_fidelity"] = len(faithful)

    called = out_df[out_df["executor_called"] == True]  # noqa: E712
    metrics["executor_json_valido_pct"] = (
        100.0 * called["executor_json_valid"].sum() / len(called) if len(called) else float("nan")
    )

    for cat in ("correto", "correto_por_acaso", "plano_invalido", "plano_errado", "erro_execucao"):
        metrics[f"cat_{cat}_pct"] = 100.0 * (out_df["error_category"] == cat).sum() / n

    for status, cnt in out_df.loc[~executable, "ref_status"].value_counts().items():
        metrics[f"plano_falha_{status}"] = int(cnt)
    return metrics


# ---------------------------------------------------------------------------
# Configuracao AutoGen
# ---------------------------------------------------------------------------

def _base_llm_config(temperature: float, model: str = OLLAMA_MODEL) -> dict:
    return {
        "config_list": [{
            "model": model,
            "base_url": OLLAMA_BASE_URL,
            "api_key": OLLAMA_API_KEY,
        }],
        "temperature": temperature,
        "timeout": TIMEOUT_SECONDS,
        "cache_seed": None,  # desliga cache do autogen p/ nao mascarar tokens/custos
    }



def _usage_totals(agents: List) -> Dict[str, int]:
    """Soma prompt/completion/total tokens de uma lista de agentes autogen,
    usando autogen.gather_usage_summary (cobre todos os modelos usados)."""
    summary = autogen.gather_usage_summary(agents)
    totals = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    usage = summary.get("usage_including_cached_inference", {})
    for model_name, stats in usage.items():
        if model_name == "total_cost":
            continue
        totals["prompt_tokens"] += stats.get("prompt_tokens", 0)
        totals["completion_tokens"] += stats.get("completion_tokens", 0)
        totals["total_tokens"] += stats.get("total_tokens", 0)
    return totals




# ---------------------------------------------------------------------------
# Montagem da entrada (QUESTION / TEXT / TABLE / HISTORY) -- identica ao E0/E1
# ---------------------------------------------------------------------------

_FIRST_TURN_RE = re.compile(r"(?i)(?:^|\n)[ \t]*question[ \t]*:")
_TURN_RE = re.compile(
    r"(?is)question[ \t]*:[ \t]*(.*?)(?:\n[ \t]*answer[ \t]*:[ \t]*(.*?))?[ \t]*(?=\n[ \t]*question[ \t]*:|\Z)")


def _cell(row: pd.Series, col: Optional[str]) -> Optional[str]:
    """Le uma coluna opcional do CSV; devolve None se nao existir/vazia."""
    if not col or col not in row.index:
        return None
    val = row[col]
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return None
    s = str(val).strip()
    return s or None


def split_dialogue(full_text: str) -> Tuple[str, List[Tuple[str, str]], Optional[str]]:
    """Separa o texto do ConvFinQA em (contexto, historico, ultima pergunta).

    No CSV, a coluna "text" traz o relatorio (texto + tabela) seguido do
    dialogo: "Question: ... Answer: ... Question: ...". A ultima pergunta
    (sem resposta) e a que deve ser respondida; os pares anteriores sao o
    historico da conversa."""
    m = _FIRST_TURN_RE.search(full_text)
    if not m:
        return full_text.strip(), [], None
    context = full_text[:m.start()].strip()
    turns = [(q.strip(), (a or "").strip()) for q, a in _TURN_RE.findall(full_text[m.start():])]
    turns = [t for t in turns if t[0]]
    if not turns:
        return context, [], None
    return context, turns[:-1], turns[-1][0]


def build_fields(row: pd.Series) -> Dict[str, str]:
    """Preenche os blocos QUESTION / TEXT / TABLE / HISTORY (igual em E0..E7).

    - Colunas separadas (QUESTION_COL, TABLE_COL, HISTORY_COL), se existirem
      no CSV, sao usadas diretamente.
    - Caso contrario, a coluna "text" e dividida por split_dialogue():
      TEXT = relatorio (texto + tabela), QUESTION = ultima pergunta do
      dialogo e HISTORY = perguntas e respostas anteriores. Na versao
      anterior o historico ia misturado no TEXT e HISTORY aparecia como
      "(none)" mesmo em conversas de varios turnos.
    - Se nao houver "Question:" no texto, tudo vai para TEXT e QUESTION
      indica que a pergunta esta nele. Nenhuma informacao e descartada.
    """
    full_text = _cell(row, TEXT_COL) or ""
    question = _cell(row, QUESTION_COL)
    table = _cell(row, TABLE_COL)
    history = _cell(row, HISTORY_COL)

    text = full_text
    if question is None:
        context, turns, last_q = split_dialogue(full_text)
        if last_q:
            text, question = context, last_q
            if history is None and turns:
                history = "\n".join(f"Question: {q}\nAnswer: {a}" for q, a in turns)
        else:
            question = "(The question is included at the end of the TEXT block.)"

    return {
        "question": question,
        "text": text if text else "(not provided)",
        "table": table if table else "(not provided separately -- any table is included in the TEXT block)",
        "history": history if history else "(none)",
    }


_NUM_RE = re.compile(r"-?\$?\d[\d,]*(?:\.\d+)?%?")


def _clean_num(tok: str) -> Optional[str]:
    t = tok.replace("$", "").replace(",", "").replace("%", "").strip()
    return t if _to_float(t) is not None else None


def _is_year_like(tok: str) -> bool:
    t = tok.replace("$", "").replace(",", "")
    return bool(re.fullmatch(r"(19|20)\d{2}", t))


def extract_direct_answer(raw: str) -> Tuple[Optional[str], str]:
    """Le a resposta do LLM direto. O prompt E0 pede "Return only the final
    answer" (texto livre), entao a leitura segue esta ordem:
      1. JSON com campo "answer" (se o modelo produzir JSON por conta propria)
      2. resposta inteira e so um numero (ex.: "25", "25%", "$1,200")
      3. numero logo apos "answer is" / "answer:" / "="
      4. ultimo numero com "%" (perguntas de percentual)
      5. ultimo numero que nao parece um ano (1900-2099)
      6. ultimo numero qualquer
    Devolve (resposta, modo_de_extracao)."""
    text = (raw or "").strip()
    if not text:
        return None, "nenhum"

    parsed = _parse_json_response(text)
    if not parsed.get("parse_error") and parsed.get("answer") is not None:
        return str(parsed.get("answer")), "json"

    stripped = re.sub(r"(?i)\b(million|billion|thousand|percent)\b", "", text).strip(" .\n\t*")
    if _clean_num(stripped) is not None:
        return _clean_num(stripped), "numero_puro"

    m = re.search(r"(?i)(?:final answer|answer)\s*(?:is|:|=)?\s*[:=]?\s*(-?\$?\d[\d,]*(?:\.\d+)?%?)", text)
    if m and _clean_num(m.group(1)) is not None:
        return _clean_num(m.group(1)), "apos_answer"

    tokens = _NUM_RE.findall(text)
    pct = [t for t in tokens if t.endswith("%")]
    if pct:
        return _clean_num(pct[-1]), "percentual"
    non_year = [t for t in tokens if not _is_year_like(t)]
    if non_year:
        return _clean_num(non_year[-1]), "ultimo_numero_nao_ano"
    if tokens:
        return _clean_num(tokens[-1]), "ultimo_numero"
    return None, "nenhum"


def _as_text(x) -> str:
    if x is None:
        return ""
    if isinstance(x, (list, dict)):
        return json.dumps(x, ensure_ascii=False)
    return str(x)


def _is_filled(x) -> bool:
    if x is None:
        return False
    if isinstance(x, (list, dict)):
        return len(x) > 0
    return str(x).strip() not in ("", "null", "None")


# ---------------------------------------------------------------------------
# Programa simbolico: normalizacao e executor Python de REFERENCIA
# ---------------------------------------------------------------------------
# O executor de referencia NAO produz a resposta do E2 (quem responde e o
# LLM Executor). Ele existe apenas para MEDIR: (a) se o plano e executavel,
# (b) se o plano esta correto (Plan Accuracy objetiva) e (c) se o LLM
# executou fielmente o plano (Execution Fidelity). No E3 esta mesma logica
# passa a ser a calculadora que produz a resposta.

def normalize_plan(parsed: dict) -> dict:
    """Mantem apenas variables / steps / unit do JSON do Planner."""
    variables = parsed.get("variables")
    steps = parsed.get("steps")
    return {
        "variables": variables if isinstance(variables, dict) else {},
        "steps": steps if isinstance(steps, list) else [],
        "unit": _as_text(parsed.get("unit")),
    }


def _apply_op(op: str, vals: List[float]) -> float:
    if op == "value":
        return vals[0]
    if op == "add":
        return sum(vals)
    if op == "mul":
        out = 1.0
        for v in vals:
            out *= v
        return out
    a, b = vals
    if op == "sub":
        return a - b
    if op == "div":
        return a / b
    if op == "percentage":
        return a / b
    if op == "percentage_change":
        return (b - a) / a
    raise ValueError(op)


_ARITY = {"value": (1, 1), "add": (2, None), "mul": (2, None), "sub": (2, 2),
          "div": (2, 2), "percentage": (2, 2), "percentage_change": (2, 2)}


def execute_program(plan: dict) -> Tuple[Optional[float], str]:
    """Calculadora: executa o programa deterministicamente. Devolve
    (resultado, status).

    status: ok | sem_steps | variavel_nao_numerica | step_malformado |
            op_invalida | aridade_invalida | arg_nao_resolvido |
            divisao_por_zero | erro_numerico
    O resultado e o do step chamado "answer" ou, se nao houver, o do ultimo.

    Resolucao de argumentos: variavel -> resultado de passo anterior ->
    constante numerica. As VARIAVEIS tem prioridade: se um passo reutiliza
    o nome de uma variavel (proibido no prompt, mas acontece), o nome
    continua apontando para o valor extraido do contexto.
    """
    steps = plan.get("steps") or []
    if not steps:
        return None, "sem_steps"

    variables: Dict[str, float] = {}
    for k, v in (plan.get("variables") or {}).items():
        fv = _to_float(v)
        if fv is None:
            return None, "variavel_nao_numerica"
        variables[str(k)] = fv

    results: Dict[str, float] = {}
    last = None
    for step in steps:
        if not isinstance(step, dict):
            return None, "step_malformado"
        op = str(step.get("op", "")).strip().lower()
        args = step.get("args")
        name = str(step.get("name", "")).strip() or f"step_{len(results)}"
        if op not in ALLOWED_OPS:
            return None, "op_invalida"
        lo, hi = _ARITY[op]
        if not isinstance(args, list) or len(args) < lo or (hi is not None and len(args) > hi):
            return None, "aridade_invalida"

        values = []
        for arg in args:
            key = str(arg).strip()
            if key in variables:
                values.append(variables[key])
            elif key in results:
                values.append(results[key])
            elif _to_float(arg) is not None:
                values.append(_to_float(arg))
            else:
                return None, "arg_nao_resolvido"
        try:
            last = _apply_op(op, values)
        except ZeroDivisionError:
            return None, "divisao_por_zero"
        except (ValueError, OverflowError):
            return None, "erro_numerico"
        results[name] = last

    return results.get("answer", last), "ok"


# ---------------------------------------------------------------------------
# Prompts preenchidos
# ---------------------------------------------------------------------------

def build_planner_prompt(row: pd.Series) -> str:
    template = E2_PLANNER_PROMPT + "\n" + OPERATION_SEMANTICS + "\n"
    if INCLUDE_PLANNER_EXAMPLE:
        template += PLANNER_EXAMPLE + "\n"
    template += INPUT_BLOCKS
    return template.format(**build_fields(row))


def build_executor_prompt(plan: dict) -> str:
    return EXECUTOR_PROMPT.format(
        semantics=OPERATION_SEMANTICS,
        program=json.dumps(plan, ensure_ascii=False, indent=2),
    )


def extract_executor_answer(raw: str) -> Tuple[bool, Optional[str], str]:
    """Le a resposta do LLM Executor. Devolve (json_valido, resposta, modo).
    Mesma logica do E1: JSON -> limpeza de "answer" textual -> fallback
    no texto bruto com extract_direct_answer (a funcao do E0)."""
    parsed = _parse_json_response(raw)
    if not parsed.get("parse_error"):
        ans = parsed.get("answer")
        if ans is not None and str(ans).strip() != "":
            if _to_float(ans) is not None:
                return True, str(_to_float(ans)), "json"
            cleaned, _ = extract_direct_answer(str(ans))
            if cleaned is not None:
                return True, cleaned, "json_texto_limpo"
        return True, None, "json_sem_answer"
    answer, mode = extract_direct_answer(raw)
    return False, answer, f"fallback_{mode}"


# ---------------------------------------------------------------------------
# Configuracao dos agentes
# ---------------------------------------------------------------------------

def build_agents():
    planner = AssistantAgent(
        name="PoT_Planner",
        system_message=SYSTEM_PERSONA,
        llm_config=_base_llm_config(LLM_TEMPERATURE),
        human_input_mode="NEVER",
        max_consecutive_auto_reply=1,
        code_execution_config=False,
    )
    executor = AssistantAgent(
        name="LLM_Executor",
        system_message=SYSTEM_PERSONA,
        llm_config=_base_llm_config(LLM_TEMPERATURE),
        human_input_mode="NEVER",
        max_consecutive_auto_reply=1,
        code_execution_config=False,
    )
    return planner, executor


# ---------------------------------------------------------------------------
# Fluxo E2: Contexto -> LLM Planner -> PoT JSON -> LLM Executor -> Resposta
# ---------------------------------------------------------------------------

def _categorize(correct: bool, executable: bool, plan_correct: bool) -> str:
    if correct:
        return "correto" if plan_correct else "correto_por_acaso"
    if not executable:
        return "plano_invalido"
    if not plan_correct:
        return "plano_errado"
    return "erro_execucao"


def run_pot(planner, executor, row: pd.Series, ground_truth: str,
            verbose: bool = False) -> PoTResult:
    result = PoTResult(question_id=None, ground_truth=str(ground_truth))
    t0 = time.perf_counter()

    # --- 1) Planner gera o programa simbolico ---
    reply = planner.generate_reply(messages=[{"role": "user", "content": build_planner_prompt(row)}])
    result.llm_calls += 1
    result.planner_raw = _extract_content(reply)
    parsed = _parse_json_response(result.planner_raw)
    result.plan_json_valid = not parsed.get("parse_error", False)
    result.plan_json_puro = is_pure_json(result.planner_raw)

    if result.plan_json_valid:
        plan = normalize_plan(parsed)
        result.plan = json.dumps(plan, ensure_ascii=False)
        result.plan_unit = plan["unit"]
        result.plan_n_steps = len(plan["steps"])

        # --- executor de REFERENCIA (so medicao) ---
        ref_answer, ref_status = execute_program(plan)
        result.ref_answer, result.ref_status = ref_answer, ref_status
        if ref_status == "ok":
            result.plan_correct = _answers_match(ref_answer, ground_truth)
            result.plan_correct_escala_pct = answers_match_pct_scale(
                ref_answer, ground_truth, build_fields(row)["question"])

        # --- 2) LLM Executor executa o programa ---
        ex_reply = executor.generate_reply(messages=[{"role": "user", "content": build_executor_prompt(plan)}])
        result.llm_calls += 1
        result.executor_called = True
        result.executor_raw = _extract_content(ex_reply)
        ex_json_ok, answer, mode = extract_executor_answer(result.executor_raw)
        result.executor_json_valid = ex_json_ok
        result.executor_json_puro = is_pure_json(result.executor_raw)
        if ex_json_ok:
            result.executor_answer_float = is_float_answer(
                _parse_json_response(result.executor_raw).get("answer"))
        result.formato_conforme = result.executor_json_puro and result.executor_answer_float
    else:
        # Sem programa nao ha o que executar: o Executor nao e chamado.
        result.ref_status = "plano_json_invalido"
        answer, mode = None, "sem_plano"

    result.latency_seconds = time.perf_counter() - t0
    result.prediction = str(answer)
    result.answer_extraction = mode
    result.parse_error = answer is None
    result.correct = _answers_match(result.prediction, ground_truth)
    result.correct_exato = answers_match_exact(result.prediction, ground_truth) if answer is not None else False
    if result.ref_status == "ok" and answer is not None:
        result.execution_faithful = _answers_match(answer, result.ref_answer)
    result.error_category = _categorize(result.correct, result.ref_status == "ok", result.plan_correct)

    if verbose:
        print(f"    [PoT_Planner] json_valido={result.plan_json_valid} steps={result.plan_n_steps} "
              f"ref={result.ref_answer} ({result.ref_status})")
        print(f"    [LLM_Executor] -> {result.prediction} (extracao: {mode}) | "
              f"categoria: {result.error_category}")
    return result


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def _fmt(x, spec=".2f"):
    try:
        return "N/D" if x is None or (isinstance(x, float) and math.isnan(x)) else format(x, spec)
    except (TypeError, ValueError):
        return str(x)


def main():
    df = pd.read_csv(CSV_PATH)
    subset = df.head(MAX_ITEMS)

    results: List[PoTResult] = []
    n_correct = 0

    print(f"Experimento: {EXPERIMENT_ID}")
    print(f"Modelo: {OLLAMA_MODEL} | Endpoint: {OLLAMA_BASE_URL}")
    print(f"Processando {len(subset)} de {len(df)} linhas do CSV (MAX_ITEMS={MAX_ITEMS})")
    print(f"PoT simbolico: LLM Planner + LLM Executor (temperatura {LLM_TEMPERATURE}) | "
          f"2 chamadas por pergunta | exemplo no prompt: {INCLUDE_PLANNER_EXAMPLE} | "
          "sem critico, sem calculadora\n")

    for i, (idx, row) in enumerate(subset.iterrows(), start=1):
        ground_truth = row[GOLD_COL]

        print(f"[{i}/{len(subset)}] Processando linha id={idx} ...")

        planner, executor = build_agents()
        agents = [planner, executor]
        usage_before = _usage_totals(agents)

        try:
            result = run_pot(planner, executor, row, ground_truth, verbose=VERBOSE)
        except Exception as e:
            print(f"  ERRO: {e}", file=sys.stderr)
            continue

        usage_after = _usage_totals(agents)
        result.question_id = idx
        result.prompt_tokens = usage_after["prompt_tokens"] - usage_before["prompt_tokens"]
        result.completion_tokens = usage_after["completion_tokens"] - usage_before["completion_tokens"]
        result.total_tokens = usage_after["total_tokens"] - usage_before["total_tokens"]

        results.append(result)
        n_correct += int(result.correct)

        print(f"  -> prediction: {result.prediction} (correto: {result.correct}) | "
              f"gt: {result.ground_truth} | tokens: {result.total_tokens} | "
              f"latencia: {result.latency_seconds:.2f}s\n")

        time.sleep(0.1)

    # ------------------------------------------------------------------
    # Salva resultados detalhados (1 linha por pergunta)
    # ------------------------------------------------------------------
    out_df = pd.DataFrame([{
        "experiment": EXPERIMENT_ID,
        "question_id": r.question_id,
        "ground_truth": r.ground_truth,
        "prediction": r.prediction,
        "correct_exato": r.correct_exato,
        "correct": r.correct,
        "answer_extraction": r.answer_extraction,
        "parse_error": r.parse_error,
        "error_category": r.error_category,
        "plan_json_valid": r.plan_json_valid,
        "plan_json_puro": r.plan_json_puro,
        "plan": r.plan,
        "plan_unit": r.plan_unit,
        "plan_n_steps": r.plan_n_steps,
        "ref_status": r.ref_status,
        "ref_answer": r.ref_answer,
        "plan_correct": r.plan_correct,
        "plan_correct_escala_pct": r.plan_correct_escala_pct,
        "execution_faithful": r.execution_faithful,
        "executor_called": r.executor_called,
        "executor_json_valid": r.executor_json_valid,
        "executor_json_puro": r.executor_json_puro,
        "executor_answer_float": r.executor_answer_float,
        "formato_conforme": r.formato_conforme,
        "planner_raw": r.planner_raw,
        "executor_raw": r.executor_raw,
        "prompt_tokens": r.prompt_tokens,
        "completion_tokens": r.completion_tokens,
        "total_tokens": r.total_tokens,
        "latency_seconds": r.latency_seconds,
        "llm_calls": r.llm_calls,
    } for r in results])
    out_df = add_question_columns(out_df, subset)
    out_df.to_csv(OUT_PATH, index=False)

    # Registro por exemplo no formato pedido no documento de ablacao.
    with open(PER_EXAMPLE_JSONL_PATH, "w", encoding="utf-8") as f:
        for r in results:
            gold_f, pred_f = _to_float(r.ground_truth), _to_float(r.prediction)
            f.write(json.dumps({
                "id": str(r.question_id),
                "gold": gold_f if gold_f is not None else r.ground_truth,
                "prediction": pred_f if pred_f is not None else (None if r.prediction == "None" else r.prediction),
                "correct": bool(r.correct),
                "latency": round(r.latency_seconds, 3),
                "llm_calls": r.llm_calls,
            }, ensure_ascii=False) + "\n")

    n = len(results)

    print("=" * 70)
    print(f"RESUMO - {EXPERIMENT_ID}: PoT simbolico + LLM Executor (AutoGen) -- ponto flutuante")
    print("=" * 70)
    if n == 0:
        print("Nenhuma linha processada com sucesso.")
        return

    total_prompt_tokens = sum(r.prompt_tokens for r in results)
    total_completion_tokens = sum(r.completion_tokens for r in results)
    total_tokens = sum(r.total_tokens for r in results)
    n_parse_errors = sum(int(r.parse_error) for r in results)
    total_latency = sum(r.latency_seconds for r in results)
    total_calls = sum(r.llm_calls for r in results)

    print(f"Linhas avaliadas:                          {n}")
    print(f"Respostas sem numero extraivel:            {n_parse_errors}/{n} "
          f"({100 * n_parse_errors / n:.2f}%)")
    print("Modo de extracao da resposta:")
    for mode, cnt in out_df["answer_extraction"].value_counts().items():
        print(f"    {mode:<32} {cnt}")

    # ------------------------------------------------------------------
    # Metricas
    # ------------------------------------------------------------------
    answer_metrics = compute_answer_level_metrics(out_df)
    squared_error_metrics = compute_squared_error_metrics(out_df)
    ci_metrics = compute_confidence_interval(out_df, confidence=0.95)
    unit_metrics = compute_unit_heuristic_accuracy(subset, out_df)
    plan_metrics = compute_plan_metrics(out_df)

    print("\n" + "-" * 70)
    print("ACERTOS")
    print("-" * 70)
    print(f"Conformidade de formato (Executor: so o JSON, com \"answer\" em float, sem tratamento): "
          f"{_fmt(plan_metrics.get('conformidade_formato_pct'))}%")
    print(f"EM exato (numero extraido == gabarito, sem tolerancia): "
          f"{_fmt(answer_metrics.get('EM_exato'))}% "
          f"[{_fmt(ci_metrics.get('EM_exato_IC_low'))}%, {_fmt(ci_metrics.get('EM_exato_IC_high'))}%]")
    print(f"Acertos: {n_correct}/{n} -> EM (tolerancia 1%) = {_fmt(answer_metrics.get('EM'))}%")
    print(f"IC {int(ci_metrics.get('IC_nivel_confianca', 0.95) * 100)}% (Wilson): "
          f"[{_fmt(ci_metrics.get('EM_IC_low'))}%, {_fmt(ci_metrics.get('EM_IC_high'))}%] "
          f"(margem +-{_fmt(ci_metrics.get('EM_IC_margem'))} p.p.)")
    print(f"EM com escala percentual: {_fmt(answer_metrics.get('EM_escala_percentual'))}% "
          f"[{_fmt(ci_metrics.get('EM_escala_pct_IC_low'))}%, {_fmt(ci_metrics.get('EM_escala_pct_IC_high'))}%] "
          f"(aceita gold x100 ou /100 nas {answer_metrics.get('n_perguntas_percentual', 0)} perguntas "
          "de percentual; use esta para comparar E0..E7)")

    print("\n" + "-" * 70)
    print("ERRO NUMERICO (apenas pares com resposta numerica valida)")
    print("-" * 70)
    print(f"MAE:                 {_fmt(answer_metrics.get('MAE'), '.4f')} "
          f"(n={answer_metrics.get('n_pares_numericos', 0)})")
    print(f"Erro relativo medio: {_fmt(answer_metrics.get('ErroRelativoMedio'))}%")
    print(f"MSE:                 {_fmt(squared_error_metrics.get('MSE'), '.4f')}")
    print(f"RMSE:                {_fmt(squared_error_metrics.get('RMSE'), '.4f')}")

    print("\n" + "-" * 70)
    print("UNIDADE (heuristico, sem LLM)")
    print("-" * 70)
    print(f"Unit accuracy heuristica: {_fmt(unit_metrics.get('unit_accuracy_heuristica'))}% "
          f"(n={unit_metrics.get('n_perguntas_com_unidade_detectada', 0)})")

    print("\n" + "-" * 70)
    print("PLANO SIMBOLICO (novo no E2)")
    print("-" * 70)
    print(f"Plano em JSON valido:         {_fmt(plan_metrics.get('plano_json_valido_pct'))}%")
    print(f"Plano executavel:             {_fmt(plan_metrics.get('plano_executavel_pct'))}%")
    print(f"Plano = so o JSON:            {_fmt(plan_metrics.get('plano_json_puro_pct'))}%")
    print(f"Plan Accuracy (objetiva):     {_fmt(plan_metrics.get('plan_accuracy_objetiva'))}%  "
          "(plano executado pela referencia Python bate com o gabarito)")
    print(f"Plan Accuracy escala %:       {_fmt(plan_metrics.get('plan_accuracy_escala_pct'))}%  "
          "(idem, aceitando gold x100 ou /100 em perguntas de percentual)")
    print(f"Execution Fidelity (LLM):     {_fmt(plan_metrics.get('execution_fidelity'))}%  "
          f"(n={plan_metrics.get('n_execution_fidelity', 0)} planos executaveis)")
    print(f"Executor em JSON valido:      {_fmt(plan_metrics.get('executor_json_valido_pct'))}%")
    print(f"Executor = so o JSON:         {_fmt(plan_metrics.get('executor_json_puro_pct'))}%")
    print(f"\"answer\" do Executor em float: {_fmt(plan_metrics.get('executor_answer_float_pct'))}%")
    falhas = {k: v for k, v in plan_metrics.items() if k.startswith("plano_falha_")}
    if falhas:
        print("Motivos de plano nao executavel:")
        for k, v in sorted(falhas.items(), key=lambda kv: -kv[1]):
            print(f"    {k.replace('plano_falha_', ''):<24} {v}")

    print("\n" + "-" * 70)
    print("DECOMPOSICAO DOS RESULTADOS (% de todas as perguntas)")
    print("-" * 70)
    print(f"correto:                      {_fmt(plan_metrics.get('cat_correto_pct'))}%")
    print(f"correto_por_acaso:            {_fmt(plan_metrics.get('cat_correto_por_acaso_pct'))}%  "
          "(acertou com plano invalido/errado)")
    print(f"plano_invalido:               {_fmt(plan_metrics.get('cat_plano_invalido_pct'))}%")
    print(f"plano_errado:                 {_fmt(plan_metrics.get('cat_plano_errado_pct'))}%")
    print(f"erro_execucao:                {_fmt(plan_metrics.get('cat_erro_execucao_pct'))}%  "
          "(plano certo, LLM errou a conta -> o que o E3 pode corrigir)")

    print("\n" + "-" * 70)
    print("CUSTO OPERACIONAL")
    print("-" * 70)
    print(f"Chamadas ao LLM:              {total_calls} ({total_calls / n:.2f} por pergunta)")
    print(f"Latencia total:               {total_latency:,.1f}s")
    print(f"Latencia media por pergunta:  {total_latency / n:,.2f}s")
    print(f"Prompt tokens (entrada):      {total_prompt_tokens:,}")
    print(f"Completion tokens (saida):    {total_completion_tokens:,}")
    print(f"Total tokens:                 {total_tokens:,}")
    print(f"Media de tokens por pergunta: {total_tokens / n:,.1f}")

    print("\n" + "-" * 70)
    print("COMPARACAO DE CUSTO HIPOTETICO (execucao real foi LOCAL/GRATUITA via Ollama)")
    print("-" * 70)
    print("Os valores abaixo simulam quanto custaria rodar o MESMO volume de "
          "tokens em uma API paga (precos de referencia, sujeitos a mudanca):\n")
    cost_metrics: Dict[str, float] = {}
    for provider, price in REFERENCE_API_PRICING.items():
        cost_input = (total_prompt_tokens / 1_000_000) * price["input_per_1m"]
        cost_output = (total_completion_tokens / 1_000_000) * price["output_per_1m"]
        cost_total = cost_input + cost_output
        cost_metrics[f"custo_hipotetico_usd__{provider}"] = cost_total
        print(f"  {provider}:")
        print(f"    ${price['input_per_1m']:.3f}/1M input, ${price['output_per_1m']:.3f}/1M output")
        print(f"    Custo estimado: ${cost_input:.4f} (input) + ${cost_output:.4f} (output) "
              f"= ${cost_total:.4f} total | ${cost_total / n:.5f} por pergunta")
    print("\n  Ollama (local, GPU/CPU proprios): $0.00 por chamada de API "
          "(custo real e apenas energia/hardware).")

    operational_metrics = {
        "experiment": EXPERIMENT_ID,
        "modelo": OLLAMA_MODEL,
        "temperatura": LLM_TEMPERATURE,
        "n_avaliadas": n,
        "n_sem_numero_extraivel": n_parse_errors,
        "llm_calls_total": total_calls,
        "llm_calls_por_pergunta": total_calls / n,
        "latencia_total_s": total_latency,
        "latencia_media_s": total_latency / n,
        "prompt_tokens_total": total_prompt_tokens,
        "completion_tokens_total": total_completion_tokens,
        "total_tokens": total_tokens,
        "tokens_medios_por_pergunta": total_tokens / n,
    }

    metrics_path = OUT_PATH.replace(".csv", "_metricas.csv")
    all_metrics = {
        **operational_metrics,
        **answer_metrics,
        **squared_error_metrics,
        **ci_metrics,
        **unit_metrics,
        **plan_metrics,
        **cost_metrics,
    }

    pd.DataFrame([all_metrics]).to_csv(metrics_path, index=False)

    print(f"\nResultados detalhados salvos em: {OUT_PATH}")
    print(f"Metricas agregadas salvas em:    {metrics_path}")
    print(f"Registro por exemplo salvo em:   {PER_EXAMPLE_JSONL_PATH}")


if __name__ == "__main__":
    main()


Experimento: E2_pot_symbolic_v4
Modelo: llama3:8b | Endpoint: http://localhost:11434/v1
Processando 100 de 100 linhas do CSV (MAX_ITEMS=100)
PoT simbolico: LLM Planner + LLM Executor (temperatura 0.2) | 2 chamadas por pergunta | exemplo no prompt: True | sem critico, sem calculadora

[1/100] Processando linha id=0 ...
[autogen.oai.client: 10-23 16:09:14] {744} WARNING - Model llama3:8b is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
[autogen.oai.client: 10-23 16:09:21] {744} WARNING - Model llama3:8b is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
    [PoT_Planner] json_valido=True steps=1 ref=93.4 (ok)
    [LLM_Executor] -> 93.4 (extracao: json) | categoria: correto
  -> prediction: 93.4 (correto: True) | gt: 93.4 | tokens: 2624 | latencia: 37.36s

[2/100] Processando linha id

In [3]:
from datetime import datetime

# Obtém a data e horário atual
agora = datetime.now()

# Exibe formatado
print("Data e horário atual:", agora.strftime("%d/%m/%Y %H:%M:%S"))

Data e horário atual: 23/10/2026 16:47:40
